# LeetCode 1630 — Arithmetic Subarrays bằng Python

Link đề: https://leetcode.com/problems/arithmetic-subarrays/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Sorting

Skeleton LeetCode:

```python
class Solution:
    def checkArithmeticSubarrays(self, nums: List[int], l: List[int], r: List[int]) -> List[bool]:
        ...
```

## 1. Mục tiêu học

1. Hiểu định nghĩa **cấp số cộng** (arithmetic sequence) và câu hỏi "**sắp xếp lại** được thành cấp số cộng không".
2. Giải mỗi truy vấn bằng **sắp xếp + kiểm tra hiệu liên tiếp** (`O(k log k)`).
3. Tối ưu xuống `O(k)` mỗi truy vấn bằng **min, max và tập hợp / mảng đánh dấu**: biết min, max và số phần tử thì cấp số cộng được xác định duy nhất.
4. Xử lý cẩn thận các trường hợp biên: mọi phần tử bằng nhau (`d = 0`), phần tử trùng lặp, `(max − min)` không chia hết cho `k − 1`.
5. Không làm thay đổi mảng đầu vào giữa các truy vấn; viết test đối chiếu với vét cạn hoán vị.

## 2. Đề bài & ví dụ

Một dãy số là **cấp số cộng** nếu có **ít nhất hai** phần tử và hiệu giữa hai phần tử liên tiếp luôn bằng nhau: `s[i+1] − s[i] == s[1] − s[0]` với mọi `i` hợp lệ. Ví dụ `1,3,5,7,9`, `7,7,7,7`, `3,−1,−5,−9` là cấp số cộng; `1,1,2,5,7` thì không.

Cho mảng `nums` gồm `n` số nguyên và hai mảng `l`, `r` gồm `m` số, mô tả `m` truy vấn: truy vấn `i` là đoạn `[l[i], r[i]]` (chỉ số từ 0, **bao gồm** hai đầu).

Trả về danh sách boolean `answer`, `answer[i] = True` nếu đoạn `nums[l[i]], …, nums[r[i]]` có thể **sắp xếp lại** thành cấp số cộng.

**Ví dụ 1:** `nums = [4,6,5,9,3,7], l = [0,0,2], r = [2,3,5]` → `[true, false, true]`.
- `[4,6,5]` → `[6,5,4]` là cấp số cộng.
- `[4,6,5,9]` không sắp được.
- `[5,9,3,7]` → `[3,5,7,9]`.

**Ví dụ 2:** `nums = [-12,-9,-3,-12,-6,15,20,-25,-20,-15,-10], l = [0,1,6,4,8,7], r = [4,4,9,7,9,10]` → `[false, true, false, false, true, true]`.

**Ràng buộc:**
- `2 ≤ n ≤ 500`, `1 ≤ m ≤ 500`
- `0 ≤ l[i] < r[i] < n` (mỗi đoạn có ít nhất 2 phần tử)
- `−10⁵ ≤ nums[i] ≤ 10⁵`

In [1]:
import random
import time
from itertools import permutations
from typing import List

examples = [
    (([4, 6, 5, 9, 3, 7], [0, 0, 2], [2, 3, 5]), [True, False, True]),
    (([-12, -9, -3, -12, -6, 15, 20, -25, -20, -15, -10], [0, 1, 6, 4, 8, 7], [4, 4, 9, 7, 9, 10]),
     [False, True, False, False, True, True]),
]
for (nums, l, r), want in examples:
    for a, b, w in zip(l, r, want):
        print(f"đoạn [{a},{b}] = {nums[a:b + 1]} -> kỳ vọng {w}")
    print()

đoạn [0,2] = [4, 6, 5] -> kỳ vọng True
đoạn [0,3] = [4, 6, 5, 9] -> kỳ vọng False
đoạn [2,5] = [5, 9, 3, 7] -> kỳ vọng True

đoạn [0,4] = [-12, -9, -3, -12, -6] -> kỳ vọng False
đoạn [1,4] = [-9, -3, -12, -6] -> kỳ vọng True
đoạn [6,9] = [20, -25, -20, -15] -> kỳ vọng False
đoạn [4,7] = [-6, 15, 20, -25] -> kỳ vọng False
đoạn [8,9] = [-20, -15] -> kỳ vọng True
đoạn [7,10] = [-25, -20, -15, -10] -> kỳ vọng True



## 3. Kiến thức nền

### 3.1 Cắt mảng (slicing) bao gồm hai đầu

`nums[a:b]` lấy chỉ số `a..b−1`. Đoạn `[l, r]` **bao gồm** `r` nên phải viết `nums[l:r + 1]`. Slicing tạo **bản sao** — sửa bản sao không ảnh hưởng `nums`.

In [2]:
nums = [4, 6, 5, 9, 3, 7]
print(nums[0:2], "vs", nums[0:2 + 1])
part = nums[2:6]; part.sort()
print("bản sao đã sắp:", part, "| nums gốc không đổi:", nums)

[4, 6] vs [4, 6, 5]
bản sao đã sắp: [3, 5, 7, 9] | nums gốc không đổi: [4, 6, 5, 9, 3, 7]


### 3.2 Kiểm tra một dãy (đã đúng thứ tự) có là cấp số cộng

Lấy `d = s[1] − s[0]` và kiểm tra mọi cặp liền kề. `zip(s, s[1:])` cho các cặp liền kề rất gọn.

In [3]:
def is_arith_in_order(s):
    d = s[1] - s[0]
    return all(b - a == d for a, b in zip(s, s[1:]))

for s in ([1, 3, 5, 7, 9], [7, 7, 7, 7], [3, -1, -5, -9], [1, 1, 2, 5, 7]):
    print(s, is_arith_in_order(s))

[1, 3, 5, 7, 9] True
[7, 7, 7, 7] True
[3, -1, -5, -9] True
[1, 1, 2, 5, 7] False


### 3.3 `sorted`, `min`, `max`, `set`

- `sorted(x)` trả list mới đã sắp, `O(k log k)`.
- `min(x)`, `max(x)` một lượt `O(k)`.
- `set(x)` bỏ trùng; `len(set(x)) < len(x)` ⇔ có phần tử lặp. Kiểm tra `v in S` trung bình `O(1)`.

In [4]:
x = [5, 9, 3, 7]
print(sorted(x), min(x), max(x), set(x), 7 in set(x), len(set([2, 2, 3])) < 3)

[3, 5, 7, 9] 3 9 {9, 3, 5, 7} True True


### 3.4 `divmod` và phép chia hết

`divmod(a, b)` trả `(a // b, a % b)`. Ta sẽ cần `d = (max − min) / (k − 1)` phải là **số nguyên** ⇒ kiểm tra phần dư bằng 0.

In [5]:
print(divmod(6, 3), divmod(7, 3), divmod(0, 4))

(2, 0) (2, 1) (0, 0)


## 4. Ý tưởng nền tảng / trực giác

Một tập `k` số sắp lại được thành cấp số cộng ⇔ **sau khi sắp tăng dần** nó là cấp số cộng (cấp số cộng giảm thì đảo ngược là tăng; công sai `d ≥ 0` sau khi sắp).

Làm tay ví dụ 1:
- `[4,6,5]` → sắp `[4,5,6]`, hiệu `1,1` ✔.
- `[4,6,5,9]` → `[4,5,6,9]`, hiệu `1,1,3` ✘.
- `[5,9,3,7]` → `[3,5,7,9]`, hiệu `2,2,2` ✔.

**Quan sát mạnh hơn:** nếu đáp án là "có", cấp số cộng phải bắt đầu ở `min`, kết thúc ở `max`, có `k` phần tử ⇒ công sai **bắt buộc** là `d = (max − min) / (k − 1)`, và các phần tử phải đúng là `min, min + d, …, max`. Ví dụ `[5,9,3,7]`: `min = 3`, `max = 9`, `k = 4` ⇒ `d = 2` ⇒ cần `{3,5,7,9}` ✔. Với `[4,6,5,9]`: `d = 5/3` không nguyên ⇒ ✘ ngay, không cần sắp.

In [6]:
for seg in ([4, 6, 5], [4, 6, 5, 9], [5, 9, 3, 7]):
    k, lo, hi = len(seg), min(seg), max(seg)
    d, rem = divmod(hi - lo, k - 1)
    need = [lo + i * d for i in range(k)] if rem == 0 else None
    print(f"{seg}: min={lo}, max={hi}, k={k}, (max-min)/(k-1) = {hi - lo}/{k - 1} -> d={d}, dư={rem}, cần {need}")

[4, 6, 5]: min=4, max=6, k=3, (max-min)/(k-1) = 2/2 -> d=1, dư=0, cần [4, 5, 6]
[4, 6, 5, 9]: min=4, max=9, k=4, (max-min)/(k-1) = 5/3 -> d=1, dư=2, cần None
[5, 9, 3, 7]: min=3, max=9, k=4, (max-min)/(k-1) = 6/3 -> d=2, dư=0, cần [3, 5, 7, 9]


## 5. Cách 1 — Brute Force / baseline: thử mọi hoán vị

**Ý tưởng:** đúng nghĩa đen của "sắp xếp lại": thử mọi hoán vị của đoạn, xem có hoán vị nào là cấp số cộng. Chỉ dùng được khi đoạn rất ngắn, nhưng là **oracle** tuyệt đối đáng tin.

In [7]:
def can_arith_perm(seg):
    return any(is_arith_in_order(p) for p in permutations(seg))

def solve_perm(nums, l, r):
    return [can_arith_perm(nums[a:b + 1]) for a, b in zip(l, r)]

print(solve_perm(*examples[0][0]), "kỳ vọng", examples[0][1])

[True, False, True] kỳ vọng [True, False, True]


**Độ phức tạp:** mỗi truy vấn `O(k! · k)` với `k = r − l + 1`. `k = 10` đã `3,6·10⁷` — không khả thi với `k = 500`.

**Nút thắt:** hầu hết hoán vị là vô ích; chỉ có **hai** thứ tự có thể là cấp số cộng (tăng dần hoặc giảm dần). ⇒ chỉ cần **sắp xếp một lần**.

In [8]:
for k in (6, 8, 9):
    seg = list(range(k, 0, -1)); seg[0] += 100             # không phải cấp số cộng -> phải thử hết
    t = time.perf_counter(); can_arith_perm(seg)
    print(f"k={k}: {time.perf_counter() - t:.3f}s")

k=6: 0.000s
k=8: 0.015s
k=9: 0.138s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chỉ cần sắp xếp.** Nếu một dãy là cấp số cộng với công sai `d`, thì thứ tự tăng dần của nó là cấp số cộng với công sai `|d|`. Ngược lại hiển nhiên. ⇒ "sắp lại được" ⇔ "bản đã sắp là cấp số cộng".

**Insight 2 — cấp số cộng được xác định bởi `(min, max, k)`.** Nếu `k` phần tử tạo được cấp số cộng (sau khi sắp) thì `d = (max − min)/(k − 1)`. Do đó:
- **Nếu** `max == min` **thì** mọi phần tử bằng nhau ⇒ luôn đúng (`d = 0`).
- **Nếu** `(max − min) % (k − 1) ≠ 0` **thì** sai.
- **Nếu không** thì đúng ⇔ tập phần tử **bằng đúng** `{min + i·d : i = 0..k−1}` — `k` giá trị phân biệt, nên **không được có phần tử trùng**.

Kiểm tra bằng `set` hoặc bằng mảng đánh dấu vị trí `(x − min)/d` — đều `O(k)`.

**Invariant khi dùng mảng đánh dấu:** mỗi phần tử `x` phải rơi vào một "ô" `idx = (x − min)/d` nguyên trong `0..k−1`, và mỗi ô nhận **đúng một** phần tử. `k` phần tử, `k` ô, không ô nào nhận hai ⇒ mọi ô đều được lấp.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp từng đoạn rồi kiểm tra hiệu

In [9]:
def check_sort(seg):
    s = sorted(seg)                      # bản sao, không đụng nums
    d = s[1] - s[0]
    for i in range(2, len(s)):
        if s[i] - s[i - 1] != d:
            return False
    return True

def solve_sort(nums, l, r):
    return [check_sort(nums[a:b + 1]) for a, b in zip(l, r)]

for (nums, l, r), want in examples:
    print(solve_sort(nums, l, r), "kỳ vọng", want)

[True, False, True] kỳ vọng [True, False, True]
[False, True, False, False, True, True] kỳ vọng [False, True, False, False, True, True]


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def checkArithmeticSubarrays(self, nums: List[int], l: List[int], r: List[int]) -> List[bool]:
        ans = []
        for a, b in zip(l, r):
            s = sorted(nums[a:b + 1])
            d = s[1] - s[0]
            ans.append(all(s[i] - s[i - 1] == d for i in range(2, len(s))))
        return ans
```

**Độ phức tạp:** Time `O(m · k log k)` với `k ≤ n` ⇒ worst `O(m · n log n)` ≈ `500 · 500 · 9`; Space `O(n)` cho bản sao.

### 7.2 Cách 3 — `min`, `max` + `set`, `O(k)` mỗi truy vấn

Từng đoạn:
1. `lo, hi = min(seg), max(seg)`; nếu `lo == hi` ⇒ `True`.
2. `d, rem = divmod(hi − lo, k − 1)`; nếu `rem ≠ 0` ⇒ `False`.
3. `S = set(seg)`; nếu `len(S) ≠ k` ⇒ có trùng ⇒ `False` (vì `d > 0` cần `k` giá trị phân biệt).
4. Kiểm tra mọi `lo + i·d` (`i = 0..k−1`) thuộc `S`.

In [10]:
def check_set(seg):
    k = len(seg)
    lo, hi = min(seg), max(seg)
    if lo == hi:
        return True
    d, rem = divmod(hi - lo, k - 1)
    if rem:
        return False
    S = set(seg)
    if len(S) != k:
        return False
    return all(lo + i * d in S for i in range(k))

def solve_set(nums, l, r):
    return [check_set(nums[a:b + 1]) for a, b in zip(l, r)]

for (nums, l, r), want in examples:
    print(solve_set(nums, l, r), "kỳ vọng", want)

[True, False, True] kỳ vọng [True, False, True]
[False, True, False, False, True, True] kỳ vọng [False, True, False, False, True, True]


Code nộp LeetCode của Cách 3:

```python
class Solution:
    def checkArithmeticSubarrays(self, nums: List[int], l: List[int], r: List[int]) -> List[bool]:
        def ok(a, b):
            seg = nums[a:b + 1]
            k, lo, hi = len(seg), min(seg), max(seg)
            if lo == hi:
                return True
            d, rem = divmod(hi - lo, k - 1)
            if rem:
                return False
            S = set(seg)
            return len(S) == k and all(lo + i * d in S for i in range(k))
        return [ok(a, b) for a, b in zip(l, r)]
```

(Bước `len(S) != k` thực ra là thừa: nếu có trùng thì `S` có ít hơn `k` giá trị nên không thể chứa đủ `k` giá trị phân biệt `lo + i·d`. Giữ lại cho dễ đọc và dừng sớm.)

**Độ phức tạp:** Time `O(m · k)` trung bình (băm), Space `O(k)`.

### 7.3 Cách 4 — `min`, `max` + mảng đánh dấu "ô" (không dùng băm)

Thay `set` bằng mảng `seen` dài `k`: mỗi `x` phải thỏa `(x − lo) % d == 0`, ô `idx = (x − lo) // d` chưa bị chiếm. Hoàn toàn tất định `O(k)`, không phụ thuộc hàm băm.

In [11]:
def check_slots(seg):
    k = len(seg)
    lo, hi = min(seg), max(seg)
    if lo == hi:
        return True
    if (hi - lo) % (k - 1):
        return False
    d = (hi - lo) // (k - 1)
    seen = [False] * k
    for x in seg:
        q, rem = divmod(x - lo, d)
        if rem or seen[q]:                 # lệch lưới hoặc trùng ô
            return False
        seen[q] = True
    return True                            # k phần tử vào k ô khác nhau => đủ

def solve_slots(nums, l, r):
    return [check_slots(nums[a:b + 1]) for a, b in zip(l, r)]

for (nums, l, r), want in examples:
    print(solve_slots(nums, l, r), "kỳ vọng", want)

[True, False, True] kỳ vọng [True, False, True]
[False, True, False, False, True, True] kỳ vọng [False, True, False, False, True, True]


Vì sao `q` luôn trong `0..k−1`? Vì `lo ≤ x ≤ hi` ⇒ `0 ≤ x − lo ≤ hi − lo = (k−1)·d` ⇒ `0 ≤ q ≤ k − 1`.

Code nộp LeetCode của Cách 4:

```python
class Solution:
    def checkArithmeticSubarrays(self, nums: List[int], l: List[int], r: List[int]) -> List[bool]:
        ans = []
        for a, b in zip(l, r):
            seg = nums[a:b + 1]
            k, lo, hi = len(seg), min(seg), max(seg)
            if lo == hi:
                ans.append(True); continue
            if (hi - lo) % (k - 1):
                ans.append(False); continue
            d = (hi - lo) // (k - 1)
            seen = [False] * k
            good = True
            for x in seg:
                q, rem = divmod(x - lo, d)
                if rem or seen[q]:
                    good = False; break
                seen[q] = True
            ans.append(good)
        return ans
```

**Độ phức tạp:** Time `O(m · k)` worst-case thật sự, Space `O(k)`.

## 8. TRACE từng bước

Trace **Cách 4** trên ví dụ 2 (6 truy vấn):

| truy vấn | đoạn | lo | hi | k | (hi−lo)%(k−1) | d | kết luận |
|---|---|---|---|---|---|---|---|
| [0,4] | −12,−9,−3,−12,−6 | −12 | −3 | 5 | 9 % 4 = 1 | — | False |
| [1,4] | −9,−3,−12,−6 | −12 | −3 | 4 | 9 % 3 = 0 | 3 | ô 1,3,0,2 ⇒ True |
| [6,9] | 20,−25,−20,−15 | −25 | 20 | 4 | 45 % 3 = 0 | 15 | −20 lệch lưới ⇒ False |
| [4,7] | −6,15,20,−25 | −25 | 20 | 4 | 0 | 15 | −6 lệch lưới ⇒ False |
| [8,9] | −20,−15 | −20 | −15 | 2 | 0 | 5 | True |
| [7,10] | −25,−20,−15,−10 | −25 | −10 | 4 | 0 | 5 | True |

In [12]:
def trace_slots(nums, l, r):
    res = []
    for a, b in zip(l, r):
        seg = nums[a:b + 1]
        k, lo, hi = len(seg), min(seg), max(seg)
        print(f"[{a},{b}] seg={seg} lo={lo} hi={hi} k={k}", end=" ")
        if lo == hi:
            print("-> mọi phần tử bằng nhau: True"); res.append(True); continue
        if (hi - lo) % (k - 1):
            print(f"-> {hi - lo} % {k - 1} = {(hi - lo) % (k - 1)} ≠ 0: False"); res.append(False); continue
        d = (hi - lo) // (k - 1)
        seen = [False] * k; good = True; log = []
        for x in seg:
            q, rem = divmod(x - lo, d)
            if rem:
                log.append(f"{x}: lệch lưới"); good = False; break
            if seen[q]:
                log.append(f"{x}: ô {q} đã có"); good = False; break
            seen[q] = True; log.append(f"{x}->ô{q}")
        print(f"d={d}: " + ", ".join(log) + f" => {good}")
        res.append(good)
    return res

assert trace_slots(*examples[1][0]) == examples[1][1]

[0,4] seg=[-12, -9, -3, -12, -6] lo=-12 hi=-3 k=5 -> 9 % 4 = 1 ≠ 0: False
[1,4] seg=[-9, -3, -12, -6] lo=-12 hi=-3 k=4 d=3: -9->ô1, -3->ô3, -12->ô0, -6->ô2 => True
[6,9] seg=[20, -25, -20, -15] lo=-25 hi=20 k=4 d=15: 20->ô3, -25->ô0, -20: lệch lưới => False
[4,7] seg=[-6, 15, 20, -25] lo=-25 hi=20 k=4 d=15: -6: lệch lưới => False
[8,9] seg=[-20, -15] lo=-20 hi=-15 k=2 d=5: -20->ô0, -15->ô1 => True
[7,10] seg=[-25, -20, -15, -10] lo=-25 hi=-10 k=4 d=5: -25->ô0, -20->ô1, -15->ô2, -10->ô3 => True


In [13]:
def trace_sort(nums, l, r):
    for a, b in zip(l, r):
        s = sorted(nums[a:b + 1])
        diffs = [y - x for x, y in zip(s, s[1:])]
        print(f"[{a},{b}] sắp = {s}, hiệu liên tiếp = {diffs} -> {len(set(diffs)) == 1}")

trace_sort(*examples[0][0])

[0,2] sắp = [4, 5, 6], hiệu liên tiếp = [1, 1] -> True
[0,3] sắp = [4, 5, 6, 9], hiệu liên tiếp = [1, 1, 3] -> False
[2,5] sắp = [3, 5, 7, 9], hiệu liên tiếp = [2, 2, 2] -> True


## 9. Code nộp LeetCode

Chọn Cách 3 (min/max + set): ngắn, `O(k)` mỗi truy vấn.

In [14]:
class Solution:
    def checkArithmeticSubarrays(self, nums: List[int], l: List[int], r: List[int]) -> List[bool]:
        def ok(a: int, b: int) -> bool:
            seg = nums[a:b + 1]
            k, lo, hi = len(seg), min(seg), max(seg)
            if lo == hi:
                return True
            d, rem = divmod(hi - lo, k - 1)
            if rem:
                return False
            S = set(seg)
            return len(S) == k and all(lo + i * d in S for i in range(k))
        return [ok(a, b) for a, b in zip(l, r)]

for (nums, l, r), want in examples:
    got = Solution().checkArithmeticSubarrays(nums, l, r)
    print(got, "OK" if got == want else "SAI")

[True, False, True] OK
[False, True, False, False, True, True] OK


## 10. Vì sao thuật toán đúng

Xét một đoạn có `k ≥ 2` phần tử, `lo = min`, `hi = max`.

1. **Sắp xếp là đủ (Cách 2).** Nếu một hoán vị là cấp số cộng công sai `d`: nếu `d ≥ 0` nó đã tăng dần (bằng bản sắp); nếu `d < 0` đảo ngược được cấp số cộng công sai `−d`, cũng là bản sắp. Chiều ngược lại hiển nhiên.
2. **Điều kiện cần (Cách 3, 4).** Nếu bản sắp là cấp số cộng `lo, lo+d, …, lo+(k−1)d` thì `hi = lo + (k−1)d` ⇒ `d = (hi−lo)/(k−1)` phải nguyên; với `d > 0` các phần tử phân biệt và đúng bằng tập `{lo + i·d}`.
3. **Điều kiện đủ.** Nếu `d` nguyên dương và tập phần tử (không trùng) bằng `{lo + i·d : 0 ≤ i < k}` thì sắp ra đúng cấp số cộng đó. Nếu `lo == hi` thì mọi phần tử bằng nhau — cấp số cộng công sai 0.
4. **Mảng đánh dấu (Cách 4).** Nếu mọi `x` vào một ô `q ∈ [0, k−1]` khác nhau thì `k` phần tử chiếm `k` ô — theo nguyên lý chuồng bồ câu là **mọi** ô ⇒ tập phần tử = `{lo + q·d}` ⇒ đúng theo (3). Nếu có `x` lệch lưới hoặc hai phần tử cùng ô thì không thể là cấp số cộng theo (2).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums) ≤ 500`, `m` = số truy vấn `≤ 500`, `k = r − l + 1 ≤ n`.

| Cách | Time mỗi truy vấn | Tổng worst | Space |
|---|---|---|---|
| 1. Hoán vị | `O(k!·k)` | không khả thi | `O(k)` |
| 2. Sắp xếp | `O(k log k)` | `O(m·n log n)` ≈ `2·10⁶` | `O(k)` |
| 3. min/max + set | `O(k)` trung bình | `O(m·n)` = `2,5·10⁵` | `O(k)` |
| 4. min/max + ô | `O(k)` worst | `O(m·n)` | `O(k)` |

Với ràng buộc nhỏ cả ba cách 2–4 đều qua thoải mái; khác biệt chỉ rõ khi `n, m` lớn.

In [15]:
rnd = random.Random(30)
nums_big = [rnd.randint(-10**5, 10**5) for _ in range(500)]
L = [0] * 500; R = [499] * 500                  # worst-case: mọi truy vấn là cả mảng
ap = list(range(-500 * 7, 0, 7)); rnd.shuffle(ap)  # một cấp số cộng bị xáo trộn
for f in (solve_sort, solve_set, solve_slots):
    for arr, name in ((nums_big, "ngẫu nhiên"), (ap, "cấp số cộng xáo")):
        t = time.perf_counter(); f(arr, L, R)
        print(f"{f.__name__:12s} {name:16s}: {time.perf_counter() - t:.3f}s")

solve_sort   ngẫu nhiên      : 0.008s


solve_sort   cấp số cộng xáo : 0.016s
solve_set    ngẫu nhiên      : 0.004s


solve_set    cấp số cộng xáo : 0.019s
solve_slots  ngẫu nhiên      : 0.004s
solve_slots  cấp số cộng xáo : 0.018s


## 12. Bộ test

- Ví dụ đề.
- Edge: đoạn 2 phần tử (luôn `True`); mọi phần tử bằng nhau; có trùng nhưng không phải tất cả bằng nhau (`[1,1,2]`, `[0,2,2,4]`); số âm; `(hi−lo)` không chia hết.
- Ngẫu nhiên seed cố định: đoạn ngắn (`k ≤ 6`) so với vét cạn hoán vị; đoạn dài đúng ràng buộc so các cách với nhau; cố ý sinh nhiều cấp số cộng xáo trộn để có đủ cả `True` lẫn `False`.
- Kiểm tra `nums` **không bị thay đổi** sau khi gọi.

In [16]:
ALL = [solve_perm, solve_sort, solve_set, solve_slots,
       lambda nums, l, r: Solution().checkArithmeticSubarrays(nums, l, r)]
for (nums, l, r), want in examples:
    for f in ALL:
        copy = nums[:]
        assert f(copy, l, r) == want, f
        assert copy == nums                         # không đổi input
edge = [
    ([5, 9], True), ([7, 7, 7, 7], True), ([1, 1, 2], False), ([0, 2, 2, 4], False),
    ([3, -1, -5, -9], True), ([1, 2, 4], False), ([10, 0, 5], True), ([100000, -100000], True),
]
for seg, want in edge:
    for chk in (can_arith_perm, check_sort, check_set, check_slots):
        assert chk(seg) == want, (chk, seg)
print("ví dụ đề + edge: OK")

ví dụ đề + edge: OK


In [17]:
def random_case(rnd, n, small=True):
    nums = []
    while len(nums) < n:
        if rnd.random() < 0.5:                       # chèn một khúc cấp số cộng (có thể xáo)
            k = rnd.randint(2, 6); a0 = rnd.randint(-20, 20); d = rnd.randint(-3, 3)
            block = [a0 + i * d for i in range(k)]; rnd.shuffle(block); nums += block
        else:
            nums.append(rnd.randint(-10, 10) if small else rnd.randint(-10**5, 10**5))
    nums = nums[:n]
    m = rnd.randint(1, 12)
    l, r = [], []
    for _ in range(m):
        a = rnd.randint(0, n - 2); b = rnd.randint(a + 1, min(n - 1, a + (5 if small else n)))
        l.append(a); r.append(b)
    return nums, l, r

rnd = random.Random(1630)
cnt_true = cnt_all = 0
for _ in range(250):
    nums, l, r = random_case(rnd, rnd.randint(2, 15))
    want = solve_perm(nums, l, r)
    cnt_true += sum(want); cnt_all += len(want)
    for f in ALL[1:]:
        assert f(nums, l, r) == want
for _ in range(30):
    nums, l, r = random_case(rnd, rnd.randint(2, 500), small=False)
    res = {tuple(f(nums, l, r)) for f in ALL[1:]}
    assert len(res) == 1
print(f"random tests: OK ({cnt_true}/{cnt_all} truy vấn nhỏ có đáp án True)")

random tests: OK (879/1602 truy vấn nhỏ có đáp án True)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên "sắp xếp lại", kiểm tra theo thứ tự gốc

In [18]:
def wrong_no_sort(nums, l, r):
    return [is_arith_in_order(nums[a:b + 1]) for a, b in zip(l, r)]       # SAI

print("sai:", wrong_no_sort(*examples[0][0]), "| đúng:", examples[0][1])

sai: [False, False, False] | đúng: [True, False, True]


**Vì sao sai:** đề hỏi có **sắp lại được** không; `[4,6,5]` theo thứ tự gốc có hiệu `2, −1`, nhưng sắp lại thành `[4,5,6]` thì được. **Sửa:** sắp trước, hoặc dùng min/max.

### Lỗi 2 — Sắp xếp **tại chỗ** trên `nums` làm hỏng các truy vấn sau

In [19]:
def wrong_inplace(nums, l, r):
    ans = []
    for a, b in zip(l, r):
        nums[a:b + 1] = sorted(nums[a:b + 1])                             # SAI: ghi đè lên nums
        s = nums[a:b + 1]
        ans.append(all(s[i] - s[i - 1] == s[1] - s[0] for i in range(2, len(s))))
    return ans

cx = ([0, 0, 5, 0], [1, 0], [3, 2])
print("sai:", wrong_inplace(list(cx[0]), cx[1], cx[2]), "| đúng:", solve_perm(*cx))

sai: [False, True] | đúng: [False, False]


**Vì sao sai:** truy vấn `[1,3]` sắp `[0,5,0]` thành `[0,0,5]` ngay trong `nums` ⇒ `nums = [0,0,0,5]`. Truy vấn `[0,2]` lẽ ra là `[0,0,5]` (sai) nhưng giờ đọc được `[0,0,0]` (đúng) ⇒ kết quả lệch. Các đoạn **chồng nhau** bị ảnh hưởng. **Sửa:** `sorted(nums[a:b+1])` vào biến mới.

### Lỗi 3 — Dùng phép chia dư mà quên kiểm tra **trùng lặp**

In [20]:
def wrong_no_dup_check(seg):
    k, lo, hi = len(seg), min(seg), max(seg)
    if lo == hi:
        return True
    if (hi - lo) % (k - 1):
        return False
    d = (hi - lo) // (k - 1)
    return all((x - lo) % d == 0 for x in seg)                           # SAI: chỉ kiểm tra "nằm trên lưới"

seg = [0, 2, 2, 6]
print("sai:", wrong_no_dup_check(seg), "| đúng:", can_arith_perm(seg))

sai: True | đúng: False


**Vì sao sai:** `[0,2,2,6]`: `d = 2`, mọi phần tử nằm trên lưới `0,2,4,6` nhưng `2` xuất hiện hai lần và **thiếu `4`**. **Sửa:** kiểm tra mỗi ô chỉ nhận một phần tử (Cách 4) hoặc kiểm tra đủ các giá trị `lo + i·d` trong `set` (Cách 3).

### Lỗi 4 — Quên trường hợp `d = 0` (mọi phần tử bằng nhau) ⇒ chia cho 0

In [21]:
def wrong_zero_d(seg):
    k, lo, hi = len(seg), min(seg), max(seg)
    if (hi - lo) % (k - 1):
        return False
    d = (hi - lo) // (k - 1)
    seen = [False] * k
    for x in seg:
        q, rem = divmod(x - lo, d)                                        # SAI khi d == 0
        if rem or seen[q]:
            return False
        seen[q] = True
    return True

try:
    print(wrong_zero_d([7, 7, 7]))
except ZeroDivisionError:
    print("ZeroDivisionError với [7, 7, 7] | đúng:", can_arith_perm([7, 7, 7]))

ZeroDivisionError với [7, 7, 7] | đúng: True


**Vì sao sai:** `[7,7,7]` có `hi − lo = 0` ⇒ `d = 0` ⇒ `divmod(…, 0)` lỗi. Hơn nữa nếu "né" bằng cách coi `d = 0` là sai thì lại trả lời sai (dãy hằng là cấp số cộng). **Sửa:** xử lý `lo == hi ⇒ True` trước.

### Lỗi 5 — Cắt thiếu phần tử cuối (`nums[l:r]`)

In [22]:
def wrong_slice(nums, l, r):
    return [check_sort(nums[a:b]) if b - a >= 2 else True for a, b in zip(l, r)]   # SAI: thiếu nums[r]

print("sai:", wrong_slice(*examples[0][0]), "| đúng:", examples[0][1])

sai: [True, True, False] | đúng: [True, False, True]


**Vì sao sai:** đoạn `[l, r]` **bao gồm** `r`; `nums[0:3]` chỉ là `nums[0..2]`. Ở truy vấn `[0,3]`, bản sai xét `[4,6,5]` (đúng là cấp số cộng) thay vì `[4,6,5,9]`. **Sửa:** `nums[l:r + 1]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Sắp xếp lại** thành …" ⇒ thứ tự ban đầu không quan trọng ⇒ nghĩ đến **sort** hoặc **đếm** (multiset).
- Cấp số cộng xác định bởi **(phần tử đầu, công sai, độ dài)** ⇒ biết `min`, `max`, `k` là biết hết ⇒ kiểm tra `O(k)`.
- Gặp phép chia cho đại lượng có thể bằng 0 (`d`, `k − 1`) ⇒ xử lý riêng trường hợp đó **trước**.
- Ràng buộc `n, m ≤ 500` ⇒ `O(m·n log n)` thừa sức; đừng tối ưu sớm (segment tree…) khi chưa cần.
- Đoạn `[l, r]` bao gồm hai đầu ⇒ `r + 1` khi cắt.

## 15. Biến thể của bài

1. **LC 1502 Can Make Arithmetic Progression From Sequence** (Easy) — chính là một truy vấn trên cả mảng.
2. **LC 413 Arithmetic Slices** — đếm đoạn con **không sắp lại** (giữ thứ tự) là cấp số cộng dài ≥ 3.
3. **LC 446 Arithmetic Slices II – Subsequence** (Hard) — dãy con (không liên tiếp).
4. **LC 1027 Longest Arithmetic Subsequence** — dãy con dài nhất.
5. **Đếm số đoạn con sắp lại được thành cấp số cộng** (bài tập 5) — dùng `gcd` của các hiệu, `O(n² log V)`.
6. **Truy vấn rất nhiều (`m` tới `10⁵`)** — cần cấu trúc dữ liệu: segment tree lưu min/max/gcd hiệu + kiểm tra phân biệt (bài khó hơn nhiều).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time / truy vấn | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị | thử mọi thứ tự | `O(k!·k)` | `O(k)` | oracle, `k ≤ 8` |
| 2. Sắp xếp | sort + hiệu bằng nhau | `O(k log k)` | `O(k)` | mặc định, dễ viết nhất |
| 3. min/max + set | công sai bắt buộc + tra tập | `O(k)` TB | `O(k)` | muốn tuyến tính, code ngắn |
| 4. min/max + ô | chuồng bồ câu, không băm | `O(k)` worst | `O(k)` | cần tất định, ngôn ngữ không có set tốt |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [1,2,4,8,6,5]`, `l = [0,1,2,3]`, `r = [1,3,4,5]`.

**Bài 2 (tính tay):** với ví dụ 2, truy vấn `[1,4]` và `[6,9]`: tính `lo`, `hi`, `k`, `d` và giải thích kết luận.

**Bài 3 (điền TODO):** hoàn thiện `common_difference(seg)` trả về công sai `d ≥ 0` nếu `seg` sắp lại được thành cấp số cộng, ngược lại trả `None`.

**Bài 4 (tự viết lại):** viết lại Cách 4 (mảng đánh dấu ô) thành một hàm duy nhất không xem code.

**Bài 5 (mini challenge):** đếm số cặp `(i, j)`, `i < j`, sao cho `nums[i..j]` sắp lại được thành cấp số cộng, trong `O(n² log V)` (gợi ý: `gcd`). Đối chiếu với cách `O(n³)` trên `n` nhỏ.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def common_difference(seg):
    # TODO: lo, hi, k; xử lý lo == hi; kiểm tra chia hết; kiểm tra đủ phần tử
    pass

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def check_slots_again(seg):
    # TODO
    pass

def count_arith_subarrays(nums):
    # TODO: với mỗi i, mở rộng j, duy trì lo, hi, set và g = gcd các hiệu
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: các đoạn `[1,2]`, `[2,4,8]`, `[4,8,6]`, `[8,6,5]`. Mức 2: sắp `[2,4,8]` có hiệu `2,4`; `[4,6,8]` hiệu `2,2`; `[5,6,8]` hiệu `1,2`. Mức 3: `[True, False, True, False]`.

**Bài 2.** Mức 1: `[1,4] = [−9,−3,−12,−6]`, `[6,9] = [20,−25,−20,−15]`. Mức 2: cả hai đều `k = 4` và `(hi − lo)` chia hết cho 3. Mức 3: `[1,4]`: `d = 3`, cần `{−12,−9,−6,−3}` ⇒ True; `[6,9]`: `d = 15`, cần `{−25,−10,5,20}` nhưng có `−20` ⇒ False.

**Bài 3.** Mức 1: như Cách 3 nhưng trả `d` thay vì `True`. Mức 2: `lo == hi ⇒ 0`. Mức 3: nhớ phân biệt `0` (hợp lệ) với `None` (không hợp lệ) khi kiểm tra kết quả.

**Bài 4.** Mức 1: `seen = [False] * k`. Mức 2: `q, rem = divmod(x − lo, d)`. Mức 3: `rem or seen[q] ⇒ False`.

**Bài 5.** Mức 1: tập `k` số phân biệt có min `lo`, max `hi` là cấp số cộng ⇔ `hi − lo == (k−1)·g` với `g = gcd` mọi hiệu `x − lo` (hoặc `x − nums[i]`). Mức 2: khi mở rộng `j`, cập nhật `g = gcd(g, |nums[j] − nums[i]|)`, `lo`, `hi`, và một `set` để phát hiện trùng. Mức 3: nếu đã có trùng mà không phải toàn bằng nhau thì mọi `j` lớn hơn cũng sai, nhưng vẫn cần duyệt để bắt trường hợp "toàn bằng nhau" — đơn giản hơn: đếm số giá trị phân biệt `c`; hợp lệ ⇔ (`c == 1`) hoặc (`c == k` và `hi − lo == (k−1)·g`).

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
b1 = Solution().checkArithmeticSubarrays([1, 2, 4, 8, 6, 5], [0, 1, 2, 3], [1, 3, 4, 5])
print("Bài 1:", b1)
assert b1 == [True, False, True, False]
nums2 = examples[1][0][0]
b2 = trace_slots(nums2, [1, 6], [4, 9])
assert b2 == [True, False]

Bài 1: [True, False, True, False]
[1,4] seg=[-9, -3, -12, -6] lo=-12 hi=-3 k=4 d=3: -9->ô1, -3->ô3, -12->ô0, -6->ô2 => True
[6,9] seg=[20, -25, -20, -15] lo=-25 hi=20 k=4 d=15: 20->ô3, -25->ô0, -20: lệch lưới => False


In [26]:
# Bài 3
def common_difference_sol(seg):
    k, lo, hi = len(seg), min(seg), max(seg)
    if lo == hi:
        return 0
    d, rem = divmod(hi - lo, k - 1)
    if rem:
        return None
    S = set(seg)
    return d if len(S) == k and all(lo + i * d in S for i in range(k)) else None

assert common_difference_sol([5, 9, 3, 7]) == 2
assert common_difference_sol([7, 7]) == 0
assert common_difference_sol([4, 6, 5, 9]) is None
rnd = random.Random(3)
for _ in range(2000):
    seg = [rnd.randint(-6, 6) for _ in range(rnd.randint(2, 6))]
    d = common_difference_sol(seg)
    assert (d is not None) == can_arith_perm(seg)
    if d is not None:
        s = sorted(seg); assert all(b - a == d for a, b in zip(s, s[1:]))
print("Bài 3: OK")

Bài 3: OK


In [27]:
# Bài 4
def check_slots_again_sol(seg):
    k, lo, hi = len(seg), min(seg), max(seg)
    if lo == hi:
        return True
    if (hi - lo) % (k - 1):
        return False
    d = (hi - lo) // (k - 1)
    seen = [False] * k
    for x in seg:
        q, rem = divmod(x - lo, d)
        if rem or seen[q]:
            return False
        seen[q] = True
    return True

rnd = random.Random(4)
for _ in range(2000):
    seg = [rnd.randint(-6, 6) for _ in range(rnd.randint(2, 6))]
    assert check_slots_again_sol(seg) == can_arith_perm(seg)
print("Bài 4: OK")

Bài 4: OK


In [28]:
# Bài 5
from math import gcd

def count_arith_subarrays_sol(nums):
    n, total = len(nums), 0
    for i in range(n):
        lo = hi = nums[i]; g = 0; seen = {nums[i]}
        for j in range(i + 1, n):
            x = nums[j]
            lo = min(lo, x); hi = max(hi, x)
            g = gcd(g, abs(x - nums[i]))
            seen.add(x)
            k = j - i + 1
            if len(seen) == 1 or (len(seen) == k and hi - lo == (k - 1) * g):
                total += 1
    return total

def count_cubic(nums):
    n = len(nums)
    return sum(check_sort(nums[i:j + 1]) for i in range(n) for j in range(i + 1, n))

print("Bài 5 ví dụ 1:", count_arith_subarrays_sol([4, 6, 5, 9, 3, 7]))
rnd = random.Random(5)
for _ in range(300):
    nums = [rnd.randint(-5, 5) for _ in range(rnd.randint(2, 14))]
    assert count_arith_subarrays_sol(nums) == count_cubic(nums)
print("Bài 5: OK")

Bài 5 ví dụ 1: 7
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template kiểm tra "sắp lại được thành cấp số cộng" trong `O(k)`:**

```python
k, lo, hi = len(seg), min(seg), max(seg)
if lo == hi: return True                 # dãy hằng
d, rem = divmod(hi - lo, k - 1)
if rem: return False                     # công sai không nguyên
S = set(seg)
return len(S) == k and all(lo + i * d in S for i in range(k))
```

**Template bản sắp xếp:** `s = sorted(seg); return all(s[i] - s[i-1] == s[1] - s[0] for i in range(2, len(s)))`

**Checklist:**
- [ ] Cắt đúng `nums[l:r + 1]` (bao gồm `r`)?
- [ ] Không sắp xếp **tại chỗ** trên `nums`?
- [ ] Xử lý `lo == hi` trước khi chia cho `d`?
- [ ] Kiểm tra `(hi − lo) % (k − 1) == 0`?
- [ ] Bắt được phần tử **trùng** (`[0,2,2,4]` phải là False)?
- [ ] Đoạn 2 phần tử luôn True?